In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedGroupKFold

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 160)

# ============================================================
# PROJECT PATH
# ============================================================

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "simulasi":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

PROCESSED_DIR = PROJECT_ROOT / "dataset" / "processed"
OUTPUT_DIR = PROJECT_ROOT / "output" / "00_split_validation"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = PROCESSED_DIR / "cimas_medication_adherence_clean.csv"

print("Project Root :", PROJECT_ROOT)
print("Dataset      :", DATA_PATH)
print("Output       :", OUTPUT_DIR)

Project Root : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent
Dataset      : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/processed/cimas_medication_adherence_clean.csv
Output       : /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/output/00_split_validation


In [2]:
df = pd.read_csv(DATA_PATH)

print("Ukuran dataset bersih:")
print(f"Observasi : {len(df):,}")
print(f"Atribut   : {df.shape[1]}")

display(df.head())

Ukuran dataset bersih:
Observasi : 8,141
Atribut   : 16


,BENEFICIARY,YEAR,Diagnosis,AGE,AGEGROUP,SCHEMETYPE,HOSPITALCOVERTYPE,ANNUALCONTRIBUTION,WELLNESSPROGRAM,GENDER,ANNUALCLAIMAMOUNT,FINALTHRESHOLD,COMPLICATIONDVELOPMENT,COMORBIDITY,COMORBIDITY.1,UNITS
0,3260976114,2022,Diabetes,16,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,MALE,219.65,ADHERENT,YES,NO,NO,69
1,3958374122,2022,Diabetes,16,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,FEMALE,928.35,ADHERENT,YES,NO,NO,397
2,3130974113,2022,Diabetes,17,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,FEMALE,581.64,NON-ADHERENT,YES,NO,NO,418
3,3278475122,2022,Diabetes,17,16-20,MEDIUMLEVEL,PRIVATE_GRADEA,900,SUBSCRIBED,FEMALE,328.61,NON-ADHERENT,NO,NO,NO,21
4,3395816142,2022,Diabetes,17,16-20,MEDIUMLEVEL,BASIC_TO_GRADEB,768,NOT_SUBSCRIBED,FEMALE,210.93,NON-ADHERENT,YES,YES,YES,42


In [3]:
df = df.copy()

df.insert(
    0,
    "OBSERVATION_ID",
    np.arange(1, len(df) + 1)
)

print("OBSERVATION_ID unik:",
      df["OBSERVATION_ID"].is_unique)

display(
    df[
        [
            "OBSERVATION_ID",
            "BENEFICIARY",
            "FINALTHRESHOLD"
        ]
    ].head()
)

OBSERVATION_ID unik: True


,OBSERVATION_ID,BENEFICIARY,FINALTHRESHOLD
0,1,3260976114,ADHERENT
1,2,3958374122,ADHERENT
2,3,3130974113,NON-ADHERENT
3,4,3278475122,NON-ADHERENT
4,5,3395816142,NON-ADHERENT


In [4]:
TARGET = "FINALTHRESHOLD"
GROUP = "BENEFICIARY"

y = df[TARGET]
groups = df[GROUP]

print("Distribusi target:")
display(
    y.value_counts()
    .rename_axis("kelas")
    .reset_index(name="jumlah")
)

print("BENEFICIARY unik:",
      groups.nunique())

Distribusi target:


,kelas,jumlah
0,NON-ADHERENT,6600
1,ADHERENT,1541


BENEFICIARY unik: 6940


In [5]:
outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

outer_splits = list(
    outer_splitter.split(
        X=df,
        y=y,
        groups=groups
    )
)

# Fold pertama digunakan sebagai Data Uji
development_idx, test_idx = outer_splits[0]

data_pengembangan = (
    df.iloc[development_idx]
    .copy()
    .reset_index(drop=True)
)

data_uji = (
    df.iloc[test_idx]
    .copy()
    .reset_index(drop=True)
)

print("Data Pengembangan :", len(data_pengembangan))
print("Data Uji           :", len(data_uji))
print("Total              :", len(data_pengembangan) + len(data_uji))

Data Pengembangan : 6512
Data Uji           : 1629
Total              : 8141


In [6]:
split_size_summary = pd.DataFrame({
    "partisi": [
        "Data Pengembangan",
        "Data Uji"
    ],
    "jumlah_observasi": [
        len(data_pengembangan),
        len(data_uji)
    ]
})

split_size_summary["persentase"] = (
    split_size_summary["jumlah_observasi"]
    / len(df)
    * 100
).round(2)

display(split_size_summary)

,partisi,jumlah_observasi,persentase
0,Data Pengembangan,6512,79.99
1,Data Uji,1629,20.01


In [7]:
def class_distribution(data, partition_name):
    result = (
        data[TARGET]
        .value_counts()
        .rename_axis("kelas")
        .reset_index(name="jumlah")
    )

    result["persentase"] = (
        result["jumlah"] /
        len(data) * 100
    ).round(2)

    result.insert(
        0,
        "partisi",
        partition_name
    )

    return result


development_class_dist = class_distribution(
    data_pengembangan,
    "Data Pengembangan"
)

test_class_dist = class_distribution(
    data_uji,
    "Data Uji"
)

outer_class_distribution = pd.concat(
    [
        development_class_dist,
        test_class_dist
    ],
    ignore_index=True
)

display(outer_class_distribution)

,partisi,kelas,jumlah,persentase
0,Data Pengembangan,NON-ADHERENT,5279,81.07
1,Data Pengembangan,ADHERENT,1233,18.93
2,Data Uji,NON-ADHERENT,1321,81.09
3,Data Uji,ADHERENT,308,18.91


In [8]:
development_groups = set(
    data_pengembangan[GROUP].unique()
)

test_groups = set(
    data_uji[GROUP].unique()
)

group_overlap = (
    development_groups
    .intersection(test_groups)
)

outer_group_summary = pd.DataFrame({
    "indikator": [
        "BENEFICIARY unik Data Pengembangan",
        "BENEFICIARY unik Data Uji",
        "BENEFICIARY overlap"
    ],
    "nilai": [
        len(development_groups),
        len(test_groups),
        len(group_overlap)
    ]
})

display(outer_group_summary)

,indikator,nilai
0,BENEFICIARY unik Data Pengembangan,5552
1,BENEFICIARY unik Data Uji,1388
2,BENEFICIARY overlap,0


In [9]:
assert len(data_pengembangan) == 6512, \
    "Jumlah Data Pengembangan berubah."

assert len(data_uji) == 1629, \
    "Jumlah Data Uji berubah."

assert len(development_groups) == 5552, \
    "Jumlah BENEFICIARY Data Pengembangan berubah."

assert len(test_groups) == 1388, \
    "Jumlah BENEFICIARY Data Uji berubah."

assert len(group_overlap) == 0, \
    "DATA LEAKAGE: BENEFICIARY muncul pada kedua partisi."

print("✓ Outer split sesuai rancangan penelitian.")

✓ Outer split sesuai rancangan penelitian.


In [10]:
split_assignment = df[
    [
        "OBSERVATION_ID",
        "BENEFICIARY",
        "FINALTHRESHOLD"
    ]
].copy()

development_ids = set(
    data_pengembangan["OBSERVATION_ID"]
)

split_assignment["partisi"] = np.where(
    split_assignment["OBSERVATION_ID"].isin(
        development_ids
    ),
    "DATA_PENGEMBANGAN",
    "DATA_UJI"
)

display(split_assignment.head())

,OBSERVATION_ID,BENEFICIARY,FINALTHRESHOLD,partisi
0,1,3260976114,ADHERENT,DATA_PENGEMBANGAN
1,2,3958374122,ADHERENT,DATA_PENGEMBANGAN
2,3,3130974113,NON-ADHERENT,DATA_PENGEMBANGAN
3,4,3278475122,NON-ADHERENT,DATA_PENGEMBANGAN
4,5,3395816142,NON-ADHERENT,DATA_PENGEMBANGAN


In [11]:
split_assignment.to_csv(
    OUTPUT_DIR / "01_outer_split_assignment.csv",
    index=False
)

In [12]:
data_pengembangan.to_csv(
    PROCESSED_DIR / "data_pengembangan.csv",
    index=False
)

data_uji.to_csv(
    PROCESSED_DIR / "data_uji.csv",
    index=False
)

print(
    "Data Pengembangan tersimpan:",
    PROCESSED_DIR / "data_pengembangan.csv"
)

print(
    "Data Uji tersimpan:",
    PROCESSED_DIR / "data_uji.csv"
)

Data Pengembangan tersimpan: /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/processed/data_pengembangan.csv
Data Uji tersimpan: /Users/riskyduha/Documents/Tesis/Prediction_Non_Adherent/dataset/processed/data_uji.csv


In [13]:
CV_SEEDS = [
    101,
    202,
    303
]

N_SPLITS = 5

print("Jumlah repetisi :", len(CV_SEEDS))
print("Jumlah fold     :", N_SPLITS)
print(
    "Total validation fold :",
    len(CV_SEEDS) * N_SPLITS
)

Jumlah repetisi : 3
Jumlah fold     : 5
Total validation fold : 15


In [14]:
cv_assignments = []

X_dev = data_pengembangan
y_dev = data_pengembangan[TARGET]
groups_dev = data_pengembangan[GROUP]

for repetition, seed in enumerate(
    CV_SEEDS,
    start=1
):

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    for fold_number, (
        train_idx,
        validation_idx
    ) in enumerate(
        splitter.split(
            X=X_dev,
            y=y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        validation_data = (
            X_dev.iloc[validation_idx]
        )

        for observation_id in validation_data[
            "OBSERVATION_ID"
        ]:

            cv_assignments.append({
                "OBSERVATION_ID": observation_id,
                "repetition": repetition,
                "seed": seed,
                "validation_fold": fold_number
            })

cv_assignment_df = pd.DataFrame(
    cv_assignments
)

display(cv_assignment_df.head(10))

,OBSERVATION_ID,repetition,seed,validation_fold
0,1,1,101,1
1,4,1,101,1
2,6,1,101,1
3,11,1,101,1
4,20,1,101,1
5,21,1,101,1
6,22,1,101,1
7,27,1,101,1
8,31,1,101,1
9,40,1,101,1


In [15]:
validation_count = (
    cv_assignment_df
    .groupby([
        "OBSERVATION_ID",
        "repetition"
    ])
    .size()
)

print(
    "Minimum validation per observasi/repetisi:",
    validation_count.min()
)

print(
    "Maksimum validation per observasi/repetisi:",
    validation_count.max()
)

assert validation_count.min() == 1
assert validation_count.max() == 1

print(
    "✓ Setiap observasi menjadi validation "
    "tepat satu kali pada setiap repetisi."
)

Minimum validation per observasi/repetisi: 1
Maksimum validation per observasi/repetisi: 1
✓ Setiap observasi menjadi validation tepat satu kali pada setiap repetisi.


In [16]:
cv_overlap_records = []

for repetition, seed in enumerate(
    CV_SEEDS,
    start=1
):

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    for fold_number, (
        train_idx,
        validation_idx
    ) in enumerate(
        splitter.split(
            X=X_dev,
            y=y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        train_data = (
            X_dev.iloc[train_idx]
        )

        validation_data = (
            X_dev.iloc[validation_idx]
        )

        train_groups = set(
            train_data[GROUP].unique()
        )

        validation_groups = set(
            validation_data[GROUP].unique()
        )

        overlap = train_groups.intersection(
            validation_groups
        )

        cv_overlap_records.append({
            "repetition": repetition,
            "seed": seed,
            "fold": fold_number,

            "training_observations":
                len(train_data),

            "validation_observations":
                len(validation_data),

            "training_beneficiaries":
                len(train_groups),

            "validation_beneficiaries":
                len(validation_groups),

            "beneficiary_overlap":
                len(overlap)
        })


cv_overlap_df = pd.DataFrame(
    cv_overlap_records
)

display(cv_overlap_df)

,repetition,seed,fold,training_observations,validation_observations,training_beneficiaries,validation_beneficiaries,beneficiary_overlap
0,1,101,1,5209,1303,4441,1111,0
1,1,101,2,5209,1303,4441,1111,0
2,1,101,3,5210,1302,4442,1110,0
3,1,101,4,5210,1302,4442,1110,0
4,1,101,5,5210,1302,4442,1110,0
5,2,202,1,5209,1303,4441,1111,0
6,2,202,2,5209,1303,4441,1111,0
7,2,202,3,5210,1302,4442,1110,0
8,2,202,4,5210,1302,4442,1110,0
9,2,202,5,5210,1302,4442,1110,0


In [17]:
max_overlap = (
    cv_overlap_df[
        "beneficiary_overlap"
    ].max()
)

print(
    "Maksimum BENEFICIARY overlap:",
    max_overlap
)

assert max_overlap == 0

print(
    "✓ Tidak terdapat BENEFICIARY overlap "
    "pada seluruh 15 training-validation fold."
)

Maksimum BENEFICIARY overlap: 0
✓ Tidak terdapat BENEFICIARY overlap pada seluruh 15 training-validation fold.


In [18]:
fold_class_records = []

for repetition, seed in enumerate(
    CV_SEEDS,
    start=1
):

    splitter = StratifiedGroupKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=seed
    )

    for fold_number, (
        train_idx,
        validation_idx
    ) in enumerate(
        splitter.split(
            X=X_dev,
            y=y_dev,
            groups=groups_dev
        ),
        start=1
    ):

        for subset_name, subset_idx in [
            ("TRAINING", train_idx),
            ("VALIDATION", validation_idx)
        ]:

            subset = (
                X_dev.iloc[subset_idx]
            )

            counts = (
                subset[TARGET]
                .value_counts()
            )

            total = len(subset)

            fold_class_records.append({
                "repetition":
                    repetition,

                "seed":
                    seed,

                "fold":
                    fold_number,

                "subset":
                    subset_name,

                "total":
                    total,

                "NON_ADHERENT":
                    int(
                        counts.get(
                            "NON-ADHERENT",
                            0
                        )
                    ),

                "ADHERENT":
                    int(
                        counts.get(
                            "ADHERENT",
                            0
                        )
                    ),

                "pct_NON_ADHERENT":
                    round(
                        counts.get(
                            "NON-ADHERENT",
                            0
                        ) / total * 100,
                        2
                    ),

                "pct_ADHERENT":
                    round(
                        counts.get(
                            "ADHERENT",
                            0
                        ) / total * 100,
                        2
                    )
            })


fold_class_distribution = pd.DataFrame(
    fold_class_records
)

display(
    fold_class_distribution.head(10)
)

,repetition,seed,fold,subset,total,NON_ADHERENT,ADHERENT,pct_NON_ADHERENT,pct_ADHERENT
0,1,101,1,TRAINING,5209,4223,986,81.07,18.93
1,1,101,1,VALIDATION,1303,1056,247,81.04,18.96
2,1,101,2,TRAINING,5209,4223,986,81.07,18.93
3,1,101,2,VALIDATION,1303,1056,247,81.04,18.96
4,1,101,3,TRAINING,5210,4223,987,81.06,18.94
5,1,101,3,VALIDATION,1302,1056,246,81.11,18.89
6,1,101,4,TRAINING,5210,4223,987,81.06,18.94
7,1,101,4,VALIDATION,1302,1056,246,81.11,18.89
8,1,101,5,TRAINING,5210,4224,986,81.07,18.93
9,1,101,5,VALIDATION,1302,1055,247,81.03,18.97


In [19]:
cv_summary = (
    cv_overlap_df
    .groupby("repetition")
    .agg(
        jumlah_fold=("fold", "count"),
        mean_training_observations=(
            "training_observations",
            "mean"
        ),
        mean_validation_observations=(
            "validation_observations",
            "mean"
        ),
        max_beneficiary_overlap=(
            "beneficiary_overlap",
            "max"
        )
    )
    .reset_index()
)

display(cv_summary)

,repetition,jumlah_fold,mean_training_observations,mean_validation_observations,max_beneficiary_overlap
0,1,5,5209.6,1302.4,0
1,2,5,5209.6,1302.4,0
2,3,5,5209.6,1302.4,0


In [20]:
split_size_summary.to_csv(
    OUTPUT_DIR /
    "02_outer_split_summary.csv",
    index=False
)

outer_class_distribution.to_csv(
    OUTPUT_DIR /
    "03_outer_class_distribution.csv",
    index=False
)

outer_group_summary.to_csv(
    OUTPUT_DIR /
    "04_outer_group_audit.csv",
    index=False
)

cv_assignment_df.to_csv(
    OUTPUT_DIR /
    "05_inner_cv_assignments.csv",
    index=False
)

cv_overlap_df.to_csv(
    OUTPUT_DIR /
    "06_inner_cv_group_overlap.csv",
    index=False
)

fold_class_distribution.to_csv(
    OUTPUT_DIR /
    "07_inner_cv_class_distribution.csv",
    index=False
)

cv_summary.to_csv(
    OUTPUT_DIR /
    "08_inner_cv_summary.csv",
    index=False
)

print("✓ Seluruh output split dan validasi tersimpan.")

✓ Seluruh output split dan validasi tersimpan.
